# Cuaderno para concatenar PDFs por carpetas con índice inicial

Este cuaderno hace lo siguiente:

1. Recorre **todas las subcarpetas** de la carpeta actual.
2. Busca en cada subcarpeta un fichero `imprimir.md`.
3. Lee las líneas de `imprimir.md`, ignorando:
   - líneas vacías
   - líneas comentadas con `#`
4. Interpreta cada línea válida como el nombre o ruta de un PDF.
5. Concatena los PDFs indicados en el orden en que aparecen.
6. Genera un PDF final llamado como la carpeta, dentro de cada carpeta procesada.
7. Añade al principio un **índice inicial** con:
   - nombre de cada PDF concatenado
   - página de inicio dentro del PDF final

Además, el PDF resultante incluye **marcadores internos (bookmarks)** para facilitar la navegación.

> **Suposición por defecto**: las rutas listadas en `imprimir.md` son relativas a la carpeta donde está ese `imprimir.md`.

In [1]:
# Instalación automática de dependencias si no están disponibles

import importlib
import subprocess
import sys

def ensure_package(import_name: str, pip_name: str | None = None) -> None:
    pip_name = pip_name or import_name
    try:
        importlib.import_module(import_name)
    except ImportError:
        print(f"Instalando {pip_name}...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", pip_name])

ensure_package("pypdf")
ensure_package("reportlab")
print("Dependencias listas.")

Dependencias listas.


In [2]:
from __future__ import annotations

import math
import os
from dataclasses import dataclass
from datetime import datetime
from io import BytesIO
from pathlib import Path
from typing import Iterable

from pypdf import PdfReader, PdfWriter
from reportlab.lib.pagesizes import A4
from reportlab.lib.units import cm
from reportlab.pdfbase.pdfmetrics import stringWidth
from reportlab.pdfgen import canvas

# ----------------------------
# Configuración
# ----------------------------

BASE_DIR = Path.cwd()               # Carpeta desde la que se ejecuta el cuaderno
IMPRIMIR_FILE = "imprimir.md"       # Nombre del fichero de entrada en cada carpeta
OUTPUT_SUFFIX = ".pdf"              # El PDF final se guarda con el nombre de la carpeta
INCLUDE_BOOKMARKS = True            # Añade marcadores internos al PDF final
PAGE_NUMBER_FONT_SIZE = 9           # Tamaño del número de página en el pie
SKIP_HIDDEN_DIRS = True             # Ignora carpetas ocultas
VERBOSE = True

print(f"Carpeta base: {BASE_DIR}")

Carpeta base: /home/adolfo/Proyectos/examenes-pau-madrid


In [3]:
@dataclass
class PdfEntry:
    source_path: Path
    display_name: str
    pages: int


@dataclass
class IndexLink:
    index_page: int
    rect: tuple[float, float, float, float]
    target_page_index: int


def list_target_folders(base_dir: Path) -> list[Path]:
    """Devuelve las subcarpetas inmediatas con imprimir.txt, ordenadas alfabéticamente."""
    folders = []
    for item in sorted(base_dir.iterdir(), key=lambda p: p.name.lower()):
        if not item.is_dir():
            continue
        if SKIP_HIDDEN_DIRS and item.name.startswith("."):
            continue
        if not (item / IMPRIMIR_FILE).exists():
            continue
        folders.append(item)
    return folders


def build_output_path(folder: Path) -> Path:
    """Construye la ruta del PDF de salida con el nombre de la carpeta."""
    return folder / f"{folder.name}{OUTPUT_SUFFIX}"


def parse_imprimir_file(imprimir_path: Path) -> list[Path]:
    """Lee imprimir.txt e ignora líneas vacías o comentadas con #.

    También tolera comentarios al final de la línea:
        documento.pdf  # comentario
    """
    pdf_paths: list[Path] = []

    with imprimir_path.open("r", encoding="utf-8") as f:
        for raw_line in f:
            line = raw_line.strip()

            if not line:
                continue
            if line.startswith("#"):
                continue

            # Permite comentarios al final de línea
            if "#" in line:
                line = line.split("#", 1)[0].strip()

            if not line:
                continue

            pdf_paths.append(Path(line))

    return pdf_paths


def resolve_pdf_paths(folder: Path, relative_paths: list[Path]) -> tuple[list[PdfEntry], list[tuple[Path, str]]]:
    """Resuelve y valida los PDFs listados en imprimir.txt.

    Devuelve:
    - lista de PDFs válidos con su número de páginas
    - lista de incidencias (ruta, motivo)
    """
    valid_entries: list[PdfEntry] = []
    issues: list[tuple[Path, str]] = []

    for rel_path in relative_paths:
        candidate = rel_path if rel_path.is_absolute() else folder / rel_path

        if not candidate.exists():
            issues.append((candidate, "No existe"))
            continue

        if candidate.suffix.lower() != ".pdf":
            issues.append((candidate, "No es un PDF"))
            continue

        try:
            reader = PdfReader(str(candidate))
            pages = len(reader.pages)
            valid_entries.append(
                PdfEntry(
                    source_path=candidate,
                    display_name=candidate.stem,
                    pages=pages,
                )
            )
        except Exception as e:
            issues.append((candidate, f"No se pudo leer: {e}"))

    return valid_entries, issues


def chunked(seq: list, size: int) -> Iterable[list]:
    for i in range(0, len(seq), size):
        yield seq[i:i + size]


def estimate_index_pages(entries_count: int, rows_first_page: int = 28, rows_other_pages: int = 34) -> int:
    """Calcula cuántas páginas de índice hacen falta.

    La primera página deja más espacio para el título.
    """
    if entries_count <= 0:
        return 1
    if entries_count <= rows_first_page:
        return 1

    remaining = entries_count - rows_first_page
    return 1 + math.ceil(remaining / rows_other_pages)


def build_index_pdf(entries_with_start_page: list[tuple[PdfEntry, int]], folder_name: str) -> tuple[BytesIO, list[IndexLink]]:
    """Genera en memoria un PDF con el índice inicial y las zonas clicables de cada fila."""
    buffer = BytesIO()
    links: list[IndexLink] = []
    page_width, page_height = A4
    c = canvas.Canvas(buffer, pagesize=A4)

    left = 2.2 * cm
    right = page_width - 2.2 * cm
    top = page_height - 2.2 * cm
    bottom = 2.0 * cm

    rows_first_page = 28
    rows_other_pages = 34

    first_page_entries = entries_with_start_page[:rows_first_page]
    remaining_entries = entries_with_start_page[rows_first_page:]
    other_pages = list(chunked(remaining_entries, rows_other_pages))

    pages = [first_page_entries] + other_pages if entries_with_start_page else [[]]

    generation_date = datetime.now().strftime("%Y-%m-%d %H:%M")

    for page_idx, page_entries in enumerate(pages, start=1):
        # Cabecera
        c.setFont("Helvetica-Bold", 18)
        c.drawString(left, top, f"Índice - {folder_name}")

        c.setFont("Helvetica", 10)
        c.drawString(left, top - 0.7 * cm, f"Generado: {generation_date}")
        c.drawString(left, top - 1.2 * cm, f"Documentos incluidos: {len(entries_with_start_page)}")

        # Línea separadora
        c.line(left, top - 1.6 * cm, right, top - 1.6 * cm)

        # Encabezados
        y = top - 2.3 * cm
        c.setFont("Helvetica-Bold", 11)
        c.drawString(left, y, "Documento")
        c.drawRightString(right, y, "Página")
        y -= 0.45 * cm
        c.line(left, y, right, y)
        y -= 0.4 * cm

        # Filas
        c.setFont("Helvetica", 10)
        line_height = 0.55 * cm

        for entry, start_page in page_entries:
            name = entry.display_name
            max_name_width = right - left - 3.0 * cm

            while stringWidth(name, "Helvetica", 10) > max_name_width and len(name) > 4:
                name = name[:-4] + "..."

            c.drawString(left, y, name)
            c.drawRightString(right, y, str(start_page))
            links.append(
                IndexLink(
                    index_page=page_idx - 1,
                    rect=(left, y - 0.15 * cm, right, y + 0.3 * cm),
                    target_page_index=start_page - 1,
                )
            )

            # puntos de relleno
            dots_start = left + min(stringWidth(name, "Helvetica", 10) + 0.35 * cm, max_name_width + left)
            dots_end = right - 1.3 * cm
            if dots_end > dots_start:
                c.setDash(1, 2)
                c.line(dots_start, y - 0.08 * cm, dots_end, y - 0.08 * cm)
                c.setDash()

            y -= line_height
            if y < bottom:
                break

        # Pie
        c.setFont("Helvetica", 9)
        c.drawRightString(right, bottom - 0.2 * cm, f"Página de índice {page_idx} de {len(pages)}")

        c.showPage()

    c.save()
    buffer.seek(0)
    return buffer, links


def build_page_number_overlay(page_number: int, total_pages: int, page_width: float, page_height: float) -> BytesIO:
    """Genera una capa con el número de página para superponer sobre una página."""
    buffer = BytesIO()
    c = canvas.Canvas(buffer, pagesize=(page_width, page_height))

    label = f"{page_number} / {total_pages}"
    bottom_margin = 0.8 * cm

    c.setFont("Helvetica", PAGE_NUMBER_FONT_SIZE)
    c.drawCentredString(page_width / 2, bottom_margin, label)
    c.save()
    buffer.seek(0)
    return buffer


def add_page_numbers(writer: PdfWriter) -> None:
    """Añade numeración a todas las páginas del PDF final."""
    total_pages = len(writer.pages)

    for page_number, page in enumerate(writer.pages, start=1):
        page_width = float(page.mediabox.width)
        page_height = float(page.mediabox.height)
        overlay_buffer = build_page_number_overlay(page_number, total_pages, page_width, page_height)
        overlay_page = PdfReader(overlay_buffer).pages[0]
        page.merge_page(overlay_page)


def add_index_links(writer: PdfWriter, links: list[IndexLink]) -> None:
    """Añade enlaces internos a las entradas del índice ya maquetado."""
    from pypdf.annotations import Link
    from pypdf.generic import ArrayObject, Fit, NumberObject

    for link in links:
        writer.add_annotation(
            link.index_page,
            Link(
                rect=link.rect,
                border=ArrayObject([NumberObject(0), NumberObject(0), NumberObject(0)]),
                target_page_index=link.target_page_index,
                fit=Fit.fit(),
            ),
        )


def merge_pdfs_with_index(entries: list[PdfEntry], output_path: Path, folder_name: str) -> tuple[int, list[tuple[str, int]]]:
    """Concatena los PDFs y añade un índice al principio.

    Devuelve:
    - número total de páginas del PDF final
    - lista de (nombre, página_inicial) del índice final
    """
    if not entries:
        raise ValueError("No hay PDFs válidos para fusionar.")

    index_pages = estimate_index_pages(len(entries))

    entries_with_start_page: list[tuple[PdfEntry, int]] = []
    current_page = index_pages + 1  # porque el índice va al principio
    for entry in entries:
        entries_with_start_page.append((entry, current_page))
        current_page += entry.pages

    # Generar índice
    index_buffer, index_links = build_index_pdf(entries_with_start_page, folder_name)
    index_reader = PdfReader(index_buffer)

    writer = PdfWriter()

    # Añadir índice
    for page in index_reader.pages:
        writer.add_page(page)

    # Añadir los PDFs originales
    for entry, start_page in entries_with_start_page:
        reader = PdfReader(str(entry.source_path))
        for page in reader.pages:
            writer.add_page(page)

        if INCLUDE_BOOKMARKS:
            try:
                writer.add_outline_item(entry.display_name, start_page - 1)
            except Exception:
                # Si la versión de pypdf no soporta esta llamada como esperamos, no bloqueamos el proceso
                pass

    add_page_numbers(writer)
    add_index_links(writer, index_links)

    writer.add_metadata({
        "/Title": f"Fusionado con índice - {folder_name}",
        "/Author": "ChatGPT",
        "/Subject": "Concatenación automática de PDFs con índice inicial",
    })

    output_path.parent.mkdir(parents=True, exist_ok=True)
    with output_path.open("wb") as f:
        writer.write(f)

    total_pages = len(writer.pages)
    index_summary = [(entry.display_name, start_page) for entry, start_page in entries_with_start_page]
    return total_pages, index_summary


def process_folder(folder: Path) -> dict:
    """Procesa una carpeta completa."""
    imprimir_path = folder / IMPRIMIR_FILE

    if not imprimir_path.exists():
        return {
            "folder": folder,
            "status": "skipped",
            "reason": f"No existe {IMPRIMIR_FILE}",
            "output": None,
            "valid_pdfs": 0,
            "issues": [],
        }

    raw_pdf_paths = parse_imprimir_file(imprimir_path)
    valid_entries, issues = resolve_pdf_paths(folder, raw_pdf_paths)

    if not valid_entries:
        return {
            "folder": folder,
            "status": "skipped",
            "reason": "No hay PDFs válidos para fusionar",
            "output": None,
            "valid_pdfs": 0,
            "issues": issues,
        }

    output_path = build_output_path(folder)
    total_pages, index_summary = merge_pdfs_with_index(valid_entries, output_path, folder.name)

    return {
        "folder": folder,
        "status": "ok",
        "reason": "",
        "output": output_path,
        "valid_pdfs": len(valid_entries),
        "issues": issues,
        "total_pages": total_pages,
        "index_summary": index_summary,
    }

In [4]:
# Ejecución principal

results = []
folders = list_target_folders(BASE_DIR)

if not folders:
    print(f"No se han encontrado subcarpetas con {IMPRIMIR_FILE} en la carpeta actual.")
else:
    print(f"Se van a revisar {len(folders)} carpeta(s).\n")

for folder in folders:
    if VERBOSE:
        print(f"Procesando: {folder.name}")

    try:
        result = process_folder(folder)
        results.append(result)

        if result["status"] == "ok":
            print(f"  OK -> {result['output'].name} | PDFs: {result['valid_pdfs']} | Páginas: {result['total_pages']}")
        else:
            print(f"  SKIP -> {result['reason']}")

        if result["issues"]:
            print("  Incidencias detectadas:")
            for issue_path, issue_reason in result["issues"]:
                print(f"    - {issue_path}: {issue_reason}")

    except Exception as e:
        results.append({
            "folder": folder,
            "status": "error",
            "reason": str(e),
            "output": None,
            "valid_pdfs": 0,
            "issues": [],
        })
        print(f"  ERROR -> {e}")

    print()

print("Proceso terminado.")

Se van a revisar 7 carpeta(s).

Procesando: filosofia
  OK -> filosofia.pdf | PDFs: 15 | Páginas: 90

Procesando: fisica
  OK -> fisica.pdf | PDFs: 14 | Páginas: 181

Procesando: ingles
  OK -> ingles.pdf | PDFs: 13 | Páginas: 95

Procesando: lengua
  OK -> lengua.pdf | PDFs: 16 | Páginas: 101

Procesando: matematicas
  OK -> matematicas.pdf | PDFs: 14 | Páginas: 98

Procesando: quimica
  OK -> quimica.pdf | PDFs: 13 | Páginas: 101

Procesando: tecnologia
  OK -> tecnologia.pdf | PDFs: 9 | Páginas: 79

Proceso terminado.


In [7]:
# Resumen final en tabla simple

import pandas as pd

summary = []
for result in results:
    summary.append({
        "carpeta": result["folder"].name,
        "estado": result["status"],
        "pdfs_validos": result.get("valid_pdfs", 0),
        "páginas": result.get("total_pages", 0),
        "salida": str(result["output"]) if result.get("output") else "",
        "motivo": result.get("reason", ""),
    })

summary_df = pd.DataFrame(summary)
print(summary_df.to_string(index=False))
summary_df

    carpeta estado  pdfs_validos  páginas                                                                 salida motivo
  filosofia     ok            15       90     /home/adolfo/Proyectos/examenes-pau-madrid/filosofia/filosofia.pdf       
     fisica     ok            14      181           /home/adolfo/Proyectos/examenes-pau-madrid/fisica/fisica.pdf       
     ingles     ok            13       95           /home/adolfo/Proyectos/examenes-pau-madrid/ingles/ingles.pdf       
     lengua     ok            16      101           /home/adolfo/Proyectos/examenes-pau-madrid/lengua/lengua.pdf       
matematicas     ok            14       98 /home/adolfo/Proyectos/examenes-pau-madrid/matematicas/matematicas.pdf       
    quimica     ok            13      101         /home/adolfo/Proyectos/examenes-pau-madrid/quimica/quimica.pdf       
 tecnologia     ok             9       79   /home/adolfo/Proyectos/examenes-pau-madrid/tecnologia/tecnologia.pdf       


,carpeta,estado,pdfs_validos,páginas,salida,motivo
0,filosofia,ok,15,90,/home/adolfo/Proyectos/examenes-pau-madrid/fil...,
1,fisica,ok,14,181,/home/adolfo/Proyectos/examenes-pau-madrid/fis...,
2,ingles,ok,13,95,/home/adolfo/Proyectos/examenes-pau-madrid/ing...,
3,lengua,ok,16,101,/home/adolfo/Proyectos/examenes-pau-madrid/len...,
4,matematicas,ok,14,98,/home/adolfo/Proyectos/examenes-pau-madrid/mat...,
5,quimica,ok,13,101,/home/adolfo/Proyectos/examenes-pau-madrid/qui...,
6,tecnologia,ok,9,79,/home/adolfo/Proyectos/examenes-pau-madrid/tec...,


## Notas útiles

- El cuaderno genera un archivo llamado como la carpeta, por ejemplo `ingles/ingles.pdf`.
- El índice muestra el nombre base de cada PDF y su página de inicio.
- Si en `imprimir.md` aparece un PDF que no existe, el proceso **lo informa** y sigue con el resto.
- El orden de concatenación es exactamente el orden de las líneas válidas de `imprimir.md`.
- Las líneas que empiecen por `#` no se ejecutan.
- También se toleran comentarios al final de línea, por ejemplo:

```txt
tema-01.pdf
tema-02.pdf  # este sí se incluye; el comentario se ignora
# tema-03.pdf
```

## Posibles ajustes

Si quieres cambiar el sufijo del PDF de salida, modifica esta constante:

```python
OUTPUT_SUFFIX = ".pdf"
```

Si quieres procesar carpetas ocultas, cambia:

```python
SKIP_HIDDEN_DIRS = False
```